## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

## TODAY:

- Part A: We will divide our documents into CHUNKS
- Part B: We will encode our CHUNKS into VECTORS and put in Chroma
- Part C: We will visualize our vectors

### PART A: Divide our documents into chunks

In [1]:
# Standard library imports for OS interactions (paths, env vars) and pattern matching
import os
import glob
# Tokenizer library from OpenAI for counting tokens
import tiktoken
# Fundamental package for scientific computing (array manipulation)
import numpy as np
# Load environment variables (like API keys) from a .env file
from dotenv import load_dotenv
# LangChain integration for OpenAI embeddings (e.g. text-embedding-3)
from langchain_openai import OpenAIEmbeddings
# LangChain integration for ChromaDB vector store
from langchain_chroma import Chroma
# LangChain integration for Hugging Face embeddings (local/hosted models)
from langchain_huggingface import HuggingFaceEmbeddings
# Document loaders for reading text files and directories
from langchain_community.document_loaders import DirectoryLoader, TextLoader
# Utility to split long text into smaller chunks for processing (RAG)
from langchain_text_splitters import RecursiveCharacterTextSplitter
# t-SNE algorithm for dimensionality reduction (visualizing embeddings 2D/3D)
from sklearn.manifold import TSNE
# Interactive plotting library for visualizing the embedding clusters
import plotly.graph_objects as go

In [2]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


OpenAI API Key exists and begins sk-proj-


In [3]:
# How many characters in all the documents?

knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Found 76 files in the knowledge base
Total characters in knowledge base: 304,434


In [4]:
# How many tokens in all the documents?

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for gpt-4.1-nano: 63,555


In [5]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    # DirectoryLoader recursively finds all files matching the glob pattern (**/*.md) in the directory
    # and loads them using the specified loader_cls (TextLoader).
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 76 documents


In [14]:
len(documents[1].page_content)


5624

In [6]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[49]}")

Divided into 413 chunks
First chunk:

page_content='2. **Onboarding**: Implementation includes:
   - 2-week onboarding process
   - Platform training for up to 5 agency staff (3 hours total)
   - Profile optimization consultation
   - Quote integration setup (if applicable)
   - CRM integration assistance

3. **Account Management**:
   - Named account manager with monthly check-ins
   - Quarterly performance reviews
   - Lead quality monitoring and optimization
   - Competitive positioning recommendations
   - Best practices sharing from top-performing agencies

4. **Platform Updates**:
   - Regular feature enhancements
   - Mobile app updates
   - Consumer experience improvements
   - New product line additions
   - Advance notice of major changes (minimum 14 days)

5. **Marketing Assistance**:
   - Quarterly marketing strategy consultations
   - Campaign performance analysis
   - Consumer trend reports
   - Co-marketing opportunity identification

---

**Signatures:**' metadata={'sou

In [ ]:
chunks[100]

### PART B: Make vectors and store in Chroma

In Week 3, you set up a Hugging Face account and got an HF_TOKEN

At this point, you might want to add it to your `.env` file and run `load_dotenv(override=True)`

(This actually shouldn't be required).

In [8]:
# Pick an embedding model

#embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 413 documents


In [29]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 413 vectors with 3,072 dimensions in the vector store


In [30]:
collection.get(limit=1)

{'ids': ['6d95e420-1859-4f45-8b0f-e22e578a7860'],
 'embeddings': None,
 'documents': ['# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'doc_type': 'company',
   'source': 'knowledge-base\\company\\about.md'}]}

### Part C: Visualize!

In [31]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [32]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

# 1. Initialize the t-SNE model
# n_components=2: Reduce our high-dimensional vectors (e.g., 384 or 1536 dim) down to just 2 dimensions (X and Y).
# random_state=42: Fixes the random seed so the cluster positions are the same every time you run this.
tsne = TSNE(n_components=2, random_state=42)
# 2. Run the reduction
# fit_transform: First 'fits' the model to the data structure, then 'transforms' it to 2D coordinates.
# reduced_vectors is now a numpy array of shape (num_documents, 2).
reduced_vectors = tsne.fit_transform(vectors)
# 3. Create the 2D scatter plot using Plotly
fig = go.Figure(data=[go.Scatter(
    # X-axis: Select all rows (:), column 0 (the first dimension from t-SNE)
    x=reduced_vectors[:, 0],
    
    # Y-axis: Select all rows (:), column 1 (the second dimension from t-SNE)
    y=reduced_vectors[:, 1],
    
    # mode='markers': Draw individual points (dots), do not connect them with lines
    mode='markers',
    
    # marker styling:
    # size=5: Pixel size of each dot
    # color=colors: The list of colors we generated earlier (mapped to document categories)
    # opacity=0.8: Makes dots slightly transparent so we can see density if they stack on top of each other
    marker=dict(size=5, color=colors, opacity=0.8),
    
    # Custom Hover Text:
    # Uses list comprehension to create a specific label string for every single dot.
    # zip(...) iterates through document types and their content simultaneously.
    # <br> is HTML for a line break.
    # [:100] truncates the text to the first 100 chars so the tooltip isn't huge.
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    
    # hoverinfo='text': Tells Plotly to ONLY display our custom text string, hiding the raw x/y coordinates
    hoverinfo='text'
)])
# 4. Polish the chart layout
fig.update_layout(
    title='2D Chroma Vector Store Visualization',
    # Note: 'scene' parameters are typically for 3D plots. For 2D, these specific axis labels might be ignored 
    # and should ideally be passed as individual xaxis_title='x', yaxis_title='y' arguments.
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,   # Width in pixels
    height=600,  # Height in pixels
    margin=dict(r=20, b=10, l=10, t=40) # HTML-style margins: Right, Bottom, Left, Top
)
fig.show()

In [33]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {m['source']} {d[:100]}..." for t, d, m in zip(doc_types, documents, metadatas)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()